# 🧬 Disease Classification using Clinical Biomarkers

## 🎯 Objective
Evaluate whether clinical biomarkers are sufficient to discriminate between autoimmune diseases and assess the reliability of such a model  .

## 🧠 Context
autoimmune diseases present overlapping biomarker profiles, making diagnosis challenging and motivating the use of data-driven classification approaches.

## ❓ Key Questions
- Can we classify diseases using biomarkers?
- Which variables are most discriminative?
- How reliable is the model?


## Data Source

This project uses the publicly available dataset:

Mahdi, M. F., Jahani, A., & Abd, D. H. (2025).
Diagnosis of rheumatic and autoimmune diseases dataset.
Data in Brief, 60, 111623.

The dataset includes 12,085 patients collected from one hospital and three laboratories in Iraq between 2019 and 2024.

It contains demographic data and laboratory biomarkers used for diagnosing autoimmune and rheumatic diseases.

The dataset is publicly available under an open-access license.


#                                                             `data understanding`


In [12]:
import pandas as pd
df = pd.read_excel("nab_projet/RA_dataset.xlsx")
df.info()
df.describe()
df["Disease"].value_counts()
(df.isnull().mean() * 100).sort_values(ascending=False)

<class 'pandas.DataFrame'>
RangeIndex: 12085 entries, 0 to 12084
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Age         12085 non-null  int64  
 1   Gender      12085 non-null  str    
 2   ESR         10997 non-null  float64
 3   CRP         9668 non-null   float64
 4   RF          10756 non-null  float64
 5   Anti-CCP    8822 non-null   float64
 6   HLA-B27     10151 non-null  str    
 7   ANA         8339 non-null   str    
 8   Anti-Ro     9185 non-null   str    
 9   Anti-La     9064 non-null   str    
 10  Anti-dsDNA  7372 non-null   str    
 11  Anti-Sm     6888 non-null   str    
 12  C3          10393 non-null  float64
 13  C4          10031 non-null  float64
 14  Disease     12085 non-null  str    
dtypes: float64(6), int64(1), str(8)
memory usage: 1.4 MB


Anti-Sm       43.003724
Anti-dsDNA    38.998759
ANA           30.997104
Anti-CCP      27.000414
Anti-La       24.997931
Anti-Ro       23.996690
CRP           20.000000
C4            16.996276
HLA-B27       16.003310
C3            14.000827
RF            10.997104
ESR            9.002896
Gender         0.000000
Age            0.000000
Disease        0.000000
dtype: float64

## Data Structure

The dataset contains 12085 patients and 15 variables:

- 6 numerical biomarkers (CRP, ESR, RF, Anti-CCP, C3, C4)
- 8 categorical variables, including diagnostic tests and target variable

### Key Observations:

- Some variables are continuous biological measurements
- Others represent binary test results (positive/negative)
- Several variables have high missing rates

## Clinical Insight

- CRP / ESR → inflammation markers
- RF / Anti-CCP → rheumatoid arthritis indicators
- ANA → autoimmune diseases
- The dataset reflects real-world clinical testing conditions, which may introduce structured missingness.

### Important Insight:

Missing values are not uniformly distributed across variables.

Two patterns emerge:

1. Highly missing variables correspond to specific diagnostic tests (e.g., Anti-dsDNA, Anti-Sm)
2. Less missing variables correspond to general inflammation markers (e.g., ESR, RF)

This suggests that missingness may be related to the nature of the tests.

However, this hypothesis will be formally tested in the next section.

This dataset is specifically designed for early diagnosis of autoimmune diseases.

## Missing Data Dependency Analysis

We investigate whether missing values depend on:

1. The target variable (Disease)
2. Other clinical variables (e.g., CRP, ESR)
3. Other missing variables

This helps determine the missing data mechanism (MCAR, MAR, MNAR).

In [13]:
import numpy as np
from scipy.stats import chi2_contingency

results = []

for col in df.columns:

    if col in ["Disease", "Age", "Gender"]:
        continue

    missing = df[col].isna()
    table = pd.crosstab(df["Disease"], missing)

    chi2, p, dof, expected = chi2_contingency(table)

    n = table.sum().sum()
    k = min(table.shape)
    cramers_v = np.sqrt(chi2 / (n * (k - 1)))

    results.append({
        "Variable": col,
        "Chi2": round(chi2, 3),
        "p_value": round(p, 5),
        "Cramers_V": round(cramers_v, 4)
    })

results_df = pd.DataFrame(results).sort_values("Cramers_V", ascending=False)

results_df

,Variable,Chi2,p_value,Cramers_V
2,RF,18.305,0.00551,0.0389
3,Anti-CCP,12.283,0.05594,0.0319
11,C4,5.774,0.44903,0.0219
7,Anti-La,5.488,0.48291,0.0213
4,HLA-B27,5.278,0.50874,0.0209
6,Anti-Ro,3.682,0.71961,0.0175
5,ANA,3.276,0.77349,0.0165
10,C3,3.296,0.77093,0.0165
9,Anti-Sm,2.275,0.89278,0.0137
1,CRP,1.563,0.95518,0.0114


## Missing Data Dependency on Target

A chi-square test was performed to evaluate whether missing values depend on the disease.

### Results

- Most p-values are greater than 0.05
- Cramér’s V values are very low (< 0.05)

### Interpretation

- No significant association was found between missingness and disease
- Missing values appear uniformly distributed across classes

### Critical Insight

This does not imply that missingness is random in a clinical sense.

Possible explanations include:
- a standardized or pre-processed dataset
- similar testing protocols across patients

### Implications

- Missingness is not informative of the target variable in this dataset
- It may still depend on other observed variables

### Limitation

This analysis only evaluates dependency with the target and does not allow us to conclude whether missingness is MAR or MNAR.

In [14]:

# 1. Variables avec données manquantes
missing_vars = [col for col in df.columns if df[col].isnull().sum() > 0]

# 2. Prédicteurs cliniques numériques à tester
predictors = ["CRP", "ESR", "RF", "C3", "C4"]

# 3. Liste pour stocker les résultats
results = []

# 4. Boucle sur chaque variable ayant des missing
for target in missing_vars:

    # on évite d'utiliser la variable elle-même comme prédicteur
    for predictor in predictors:

        if target == predictor:
            continue

        # ignorer les prédicteurs trop incomplets
        if df[predictor].isnull().mean() > 0.30:
            continue

        # sous-dataframe avec les lignes où le prédicteur est disponible
        temp = df[[target, predictor]].copy().dropna(subset=[predictor])

        # calcul de la médiane du prédicteur
        median_val = temp[predictor].median()

        # création de deux groupes : faible / élevé
        temp["predictor_group"] = np.where(temp[predictor] > median_val, "High", "Low")

        # variable binaire de missing pour la cible
        temp["target_missing"] = temp[target].isnull().astype(int)

        # taux de missing par groupe
        missing_by_group = temp.groupby("predictor_group")["target_missing"].mean()

        low_missing = missing_by_group.get("Low", np.nan)
        high_missing = missing_by_group.get("High", np.nan)
        diff = low_missing - high_missing

        results.append({
            "Target_missing": target,
            "Predictor": predictor,
            "Predictor_median": round(median_val, 3),
            "Missing_rate_low": round(low_missing, 4),
            "Missing_rate_high": round(high_missing, 4),
            "Difference_low_minus_high": round(diff, 4),
            "Abs_difference": round(abs(diff), 4)
        })

# 5. Tableau final
results_df = pd.DataFrame(results).sort_values("Abs_difference", ascending=False)

# 6. Affichage
results_df

,Target_missing,Predictor,Predictor_median,Missing_rate_low,Missing_rate_high,Difference_low_minus_high,Abs_difference
46,Anti-Sm,C4,38.0,0.4394,0.4236,0.0158,0.0158
41,Anti-dsDNA,C4,38.0,0.3859,0.4008,-0.0149,0.0149
42,Anti-Sm,CRP,15.6,0.4262,0.4384,-0.0122,0.0122
24,ANA,RF,19.2,0.3171,0.3051,0.0120,0.0120
19,HLA-B27,RF,19.2,0.1544,0.1662,-0.0119,0.0119
10,RF,C3,133.0,0.1049,0.1153,-0.0105,0.0105
14,Anti-CCP,RF,19.2,0.2623,0.2726,-0.0103,0.0103
29,Anti-Ro,RF,19.2,0.2332,0.2425,-0.0092,0.0092
0,ESR,CRP,15.6,0.0927,0.0839,0.0088,0.0088
33,Anti-La,ESR,28.0,0.2457,0.2545,-0.0088,0.0088


## Final Missing Data Interpretation

We investigated whether missing values depend on:

- The target variable (Disease)
- Clinical predictors (CRP, ESR, RF, C3, C4)

### Findings

- No significant association with Disease (Chi² test)
- No meaningful variation across clinical predictors
- Differences in missing rates are consistently below 2%

### Interpretation

- Missingness appears non-informative in the dataset, but may still reflect underlying clinical decision processes.
- No clear pattern of clinically driven missingness is detected

### Conclusion

- Missing values can be reasonably approximated as non-informative in this dataset
- An MCAR-like assumption is acceptable for modeling purposes

### Limitation

- This conclusion is based only on observed variables
- It may not fully reflect real-world clinical processes

## Imputation Strategy

To handle missing values in continuous biomarkers, we used MICE (Multiple Imputation by Chained Equations) with BayesianRidge as the base estimator.

This choice is justified by several reasons:

- the dataset contains substantial missing values, and deleting incomplete rows would lead to significant information loss
- previous analyses suggest that missingness is approximately non-informative in this dataset
- continuous biomarkers such as ESR, CRP, RF, Anti-CCP, C3, and C4 are likely related biologically and statistically
- MICE leverages multivariate relationships, making it more appropriate than simple mean or median imputation

BayesianRidge was selected because it is stable, efficient, and suitable for continuous variables.

### Limitations

- it mainly captures linear relationships
- it may oversimplify complex non-linear interactions
- imputed values are estimates rather than true observations

For this reason, post-imputation validation is necessary to ensure that the overall structure of the data is preserved.

**Note:** MICE with BayesianRidge was applied only to numerical variables. Categorical variables were handled separately.

In [15]:
# =========================================
# 1. Imports
# =========================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import OneHotEncoder

# nécessaire pour IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


# =========================================
# 2. Définir les variables
# =========================================
target_col = "Disease"

numeric_cols = ["Age", "ESR", "CRP", "RF", "Anti-CCP", "C3", "C4"]
categorical_cols = ["Gender", "HLA-B27", "ANA", "Anti-Ro", "Anti-La", "Anti-dsDNA", "Anti-Sm"]

X = df[numeric_cols + categorical_cols].copy()
y = df[target_col].copy()


# =========================================
# 3. Split train/test AVANT imputation
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =========================================
# 4. Imputation des variables numériques
# =========================================
num_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    initial_strategy="median"
)

X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train[numeric_cols]),
    columns=numeric_cols,
    index=X_train.index
)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test[numeric_cols]),
    columns=numeric_cols,
    index=X_test.index
)


# =========================================
# 5. Gestion des variables catégorielles
# =========================================
X_train_cat = X_train[categorical_cols].copy()
X_test_cat = X_test[categorical_cols].copy()

for col in categorical_cols:
    X_train_cat[col] = X_train_cat[col].fillna("Missing")
    X_test_cat[col] = X_test_cat[col].fillna("Missing")


# =========================================
# 6. Encodage One-Hot
# =========================================
encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",  # 🔥 correction importante
    sparse_output=False
)

X_train_cat_encoded = pd.DataFrame(
    encoder.fit_transform(X_train_cat),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train_cat.index
)

X_test_cat_encoded = pd.DataFrame(
    encoder.transform(X_test_cat),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test_cat.index
)


# =========================================
# 7. Dataset final
# =========================================
X_train_final = pd.concat([X_train_num, X_train_cat_encoded], axis=1)
X_test_final = pd.concat([X_test_num, X_test_cat_encoded], axis=1)


# =========================================
# 8. Vérification
# =========================================
print("Missing train:", X_train_final.isnull().sum().sum())
print("Missing test :", X_test_final.isnull().sum().sum())

print("\nShape train:", X_train_final.shape)
print("Shape test :", X_test_final.shape)

Missing train: 0
Missing test : 0

Shape train: (9668, 20)
Shape test : (2417, 20)


## Post-Imputation Check

The preprocessing pipeline successfully removed all missing values from both training and test sets.

### Key Observations

- Numerical variables were imputed using MICE with BayesianRidge
- Categorical missing values were preserved as a distinct `"Missing"` category
- One-hot encoding increased the number of features

This confirms that the dataset is now fully usable for machine learning.

### Important Note

The imputed numerical values are estimated rather than true observations.
As a result, some values may not exactly match real measurements, which is expected with model-based imputation.

This may slightly reduce variability and should be considered when interpreting the results.

In [16]:
# ============================
# Validation post-imputation
# ============================

numeric_cols = ["Age", "ESR", "CRP", "RF", "Anti-CCP", "C3", "C4"]

# AVANT imputation (train uniquement)
before_stats = X_train[numeric_cols].describe().T[["mean", "50%", "std"]].rename(
    columns={"50%": "median"}
)

# APRÈS imputation
after_stats = X_train_num[numeric_cols].describe().T[["mean", "50%", "std"]].rename(
    columns={"50%": "median"}
)

# Comparaison
comparison_stats = before_stats.merge(
    after_stats,
    left_index=True,
    right_index=True,
    suffixes=("_before", "_after")
)

comparison_stats

,mean_before,median_before,std_before,mean_after,median_after,std_after
Age,49.903082,50.0,17.710569,49.903082,50.000000,17.710569
ESR,24.222930,28.0,14.388657,24.212716,28.000000,14.165362
CRP,13.299057,15.6,10.364106,13.298117,15.500000,10.052637
RF,19.733043,19.2,11.473322,19.724205,19.300000,10.955544
Anti-CCP,19.651717,19.1,11.610417,19.678108,19.500000,10.271112
C3,131.632857,133.0,36.444702,131.646396,133.000000,34.223815
C4,38.202935,38.0,20.001346,38.200465,38.007053,18.522682


## Post-Imputation Validation

We compared the distribution of numerical variables before and after imputation on the training set.

### Observations

- Mean values remain almost unchanged across all variables
- Median values are preserved
- Standard deviations show a slight decrease

### Interpretation

The imputation preserves the central tendency of the data while slightly reducing variability.

This behavior is expected with model-based methods such as MICE with BayesianRidge, which tend to produce smoothed estimates.

### Conclusion

The imputation strategy does not significantly distort the data distribution and is acceptable for modeling.

However, the slight reduction in variance should be considered as a limitation, as it may reduce the representation of extreme values.

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ============================
# Encodage de la target
# ============================
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


# ============================
# Modèle
# ============================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

model.fit(X_train_final, y_train_encoded)


# ============================
# Prédictions
# ============================
y_pred = model.predict(X_test_final)


# ============================
# Évaluation
# ============================
print("Accuracy :", accuracy_score(y_test_encoded, y_pred))
print("\nClassification Report:\n", classification_report(y_test_encoded, y_pred))

cm = confusion_matrix(y_test_encoded, y_pred)
cm

Accuracy : 0.8469176665287547

Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.62      0.68       425
           1       0.89      0.84      0.86       321
           2       0.87      0.87      0.87       357
           3       0.78      0.60      0.68       103
           4       0.80      0.95      0.87       570
           5       0.87      0.90      0.88       370
           6       1.00      0.98      0.99       271

    accuracy                           0.85      2417
   macro avg       0.85      0.82      0.83      2417
weighted avg       0.85      0.85      0.84      2417



array([[264,   0,  40,   5, 115,   1,   0],
       [  1, 270,   2,   1,   0,  47,   0],
       [ 36,   2, 312,   1,   6,   0,   0],
       [ 28,   0,   0,  62,  13,   0,   0],
       [ 16,   0,   4,   8, 542,   0,   0],
       [  2,  29,   1,   3,   3, 332,   0],
       [  0,   3,   0,   0,   0,   3, 265]])

In [18]:
for i, label in enumerate(label_encoder.classes_):
    print(i, "→", label)

0 → Ankylosing Spondylitis
1 → Normal
2 → Psoriatic Arthritis
3 → Reactive Arthritis
4 → Rheumatoid Arthritis
5 → Sjögren's Syndrome
6 → Systemic Lupus Erythematosus


In [ ]:
## Clinical Interpretation of Model Results

### Well-predicted diseases

- **Systemic Lupus Erythematosus** shows excellent performance (recall ≈ 0.98), likely due to specific biomarkers such as Anti-dsDNA and Anti-Sm
- **Rheumatoid Arthritis** is also well identified (recall ≈ 0.95), supported by RF and Anti-CCP
- **Sjögren's Syndrome** shows strong performance, likely driven by Anti-Ro and Anti-La markers

### Challenging diseases

- **Ankylosing Spondylitis** is frequently confused with Rheumatoid Arthritis
- **Reactive Arthritis** is also difficult to distinguish and often misclassified

### Interpretation

These confusions suggest that some diseases share overlapping inflammatory or immunological profiles.

In particular:
- inflammatory markers such as CRP and ESR are not disease-specific
- some conditions lack strong discriminative biomarkers

### Key Insight

The model highlights the limitations of using biological markers alone for differential diagnosis in autoimmune diseases.

### Conclusion

- The model performs well overall
- However, some diseases remain difficult to distinguish
- This reflects real-world clinical challenges in autoimmune diagnosis

In [19]:
import pandas as pd

feature_importance = pd.Series(
    model.feature_importances_,
    index=X_train_final.columns
).sort_values(ascending=False)

feature_importance.head(15)

ESR                 0.182974
CRP                 0.151310
RF                  0.137623
Anti-CCP            0.132987
C3                  0.109070
C4                  0.087435
Age                 0.031194
HLA-B27_Negative    0.026813
Anti-La_Negative    0.022195
ANA_Negative        0.020565
Anti-Ro_Negative    0.020060
HLA-B27_Positive    0.018138
Anti-Ro_Positive    0.010170
Anti-La_Positive    0.009226
ANA_Positive        0.008654
dtype: float64

## Feature Importance Analysis

The most important features in the model are:

- ESR and CRP (inflammation markers)
- RF and Anti-CCP (rheumatoid arthritis markers)
- Complement proteins C3 and C4

### Interpretation

The model relies heavily on general inflammation markers and key autoantibodies.

However, some disease-specific biomarkers (e.g., Anti-Ro, Anti-La, ANA) have relatively low importance.

### Insight

This suggests that the model primarily captures overall inflammatory activity rather than fine-grained immunological differences.

### Implication

- Strong signals (e.g., Rheumatoid Arthritis, Lupus) are well detected
- Diseases with less distinct biological profiles are harder to classify

### Limitation

Feature importance may be influenced by:
- variable type (continuous vs categorical)
- encoding strategy
- imputation effects

Therefore, importance values should be interpreted with caution.

## Final Conclusion

In this project, we developed a machine learning model to classify autoimmune and rheumatic diseases using clinical and biological features.

### Summary of Findings

- The Random Forest model achieved an overall accuracy of approximately 85%
- Some diseases, such as **Systemic Lupus Erythematosus** and **Rheumatoid Arthritis**, were well identified
- Other diseases, including **Ankylosing Spondylitis** and **Reactive Arthritis**, showed lower performance and were frequently misclassified

### Key Insights

- The model relies primarily on general inflammation markers (ESR, CRP) and major autoantibodies (RF, Anti-CCP)
- Disease-specific biomarkers (e.g., Anti-Ro, Anti-La, ANA) contribute less to the model than expected
- Several diseases share overlapping biological profiles, making them difficult to distinguish using available features

### Interpretation

The results highlight an important limitation:
**biomarkers alone may not be sufficient for precise differential diagnosis in autoimmune diseases.**

The model performance reflects real-world clinical challenges, where multiple conditions present similar inflammatory or immunological patterns.

### Limitations

- The dataset may not fully reflect real clinical practice (e.g., standardized testing, missingness patterns)
- Imputation may have slightly reduced data variability
- Only biological variables were used; clinical symptoms and imaging data were not included
- No external validation dataset was used

### Perspectives

Future improvements could include:

- integrating clinical features (symptoms, history)
- testing other models or feature engineering strategies
- using more representative or real-world datasets
- applying model interpretation techniques (e.g., SHAP)

### Final Statement

This project demonstrates both the potential and the limitations of machine learning in medical diagnosis.

While predictive performance is strong, the results emphasize the need for combining data-driven approaches with clinical expertise.
The model results are consistent with known clinical challenges highlighted in the literature.